# 03b — Transformer (secuencias) · CERT r4.2

**Motivación.** Los 3 modelos de `03_models.ipynb` (reglas, Isolation Forest, autoencoder) tratan cada usuario-día como una observación independiente. El **escenario 2** (robo de datos para un competidor) no es un día raro aislado: es una **escalada temporal** — el usuario navega webs de empleo, contacta al competidor y va incrementando el ritmo de copias USB durante varias semanas antes de irse. Un modelo que mire cada día por separado (autoencoder: 9/30 detectados; reglas: 1/30) tiene difícil capturar esa tendencia.

Este notebook añade un **4º modelo no supervisado**, `score_transformer`, basado en una arquitectura de secuencias (Transformer encoder) que reconstruye **ventanas temporales causales** de 28 días por usuario. El error de reconstrucción del **último día de la ventana** es el score de anomalía: alto cuando el comportamiento reciente del usuario se desvía de la trayectoria que el modelo aprendió a esperar para él/los usuarios en general.

El score se integra como una columna más en `user_day_scores.parquet`, junto a `score_rules`, `score_iforest`, `score_autoencoder`.

**Nota importante sobre la calidad de los resultados:** este notebook está diseñado para entrenar en **GPU (Colab)**. En **CPU local** solo se ejecutan 2 épocas sobre un submuestreo de ventanas de entrenamiento — sirve únicamente para **validar que el pipeline completo funciona de extremo a extremo** (shapes, dataset, entrenamiento, scoring, integración, evaluación). Los números de calidad (AUC-PR, detección por escenario) de la ejecución en CPU **no son representativos**; los resultados válidos se obtienen del run completo en Colab con GPU.

In [ ]:
# [compute]
# === Setup: entorno, dependencias y rutas (dual Colab/local, igual que 03_models) ===
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    %pip install -q polars pyarrow
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/CERT_data")
    DATA_DIR = BASE / "data" / "r4.2"
    ANSWERS_DIR = BASE / "answers"
    PROCESSED_DIR = BASE / "processed"
else:
    # Local: el notebook vive en SIEM/ — reutilizamos src/config.py
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    sys.path.insert(0, str(PROJECT_ROOT))
    from src.config import DATA_DIR, ANSWERS_DIR, PROCESSED_DIR

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

import polars as pl

DATE_FORMAT = "%m/%d/%Y %H:%M:%S"
WORK_HOURS = (8, 18)

print(f"Entorno: {'Colab' if IN_COLAB else 'Local'}")
print(f"Datos:   {DATA_DIR}")
print(f"Answers: {ANSWERS_DIR}")
print(f"Procesados: {PROCESSED_DIR}")

# --- Dependencias de modelado: instalación condicional si faltan en local ---
try:
    import numpy as np
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import average_precision_score
    import torch
    import torch.nn as nn
except ImportError:
    import subprocess, sys as _sys
    subprocess.run([_sys.executable, "-m", "pip", "install", "-q", "scikit-learn", "torch", "numpy"], check=True)
    import numpy as np
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import average_precision_score
    import torch
    import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

In [ ]:
# [compute]
# === Carga de features y construcción de la matriz de secuencia X (15 features) ===

df = pl.read_parquet(PROCESSED_DIR / "user_day_features.parquet")
df = df.sort("user", "day")

COUNT_FEATURES = [
    "n_logons", "n_afterhours_logons", "n_distinct_pcs", "n_other_pc_logons",
    "n_usb_connects", "n_afterhours_usb",
    "n_file_copies", "n_afterhours_file_copies",
    "n_emails", "n_external_emails", "total_attachments", "total_email_size", "n_recipients_total",
]

# log1p sobre los 13 conteos; is_weekend a 0/1; first_logon_hour tal cual (-1 incluido)
raw = df.select(
    *[pl.col(c).cast(pl.Float64).log1p().alias(c) for c in COUNT_FEATURES],
    pl.col("is_weekend").cast(pl.Int8).alias("is_weekend"),
    pl.col("first_logon_hour").cast(pl.Float64).alias("first_logon_hour"),
).to_numpy()

F = raw.shape[1]
assert F == 15, f"esperadas 15 features, obtenidas {F}"

scaler = StandardScaler()
X = scaler.fit_transform(raw).astype(np.float32)

print(f"df: {df.shape}")
print(f"X:  {X.shape}, F={F}")
print(f"device: {device}")

## Representación por secuencias y ventanas trailing

Cada usuario-día se representa como un vector de `F=15` features estandarizadas (`X`). Para capturar la **trayectoria temporal** de cada usuario, en lugar de mirar cada fila por separado construimos, para cada usuario-día `t`, una **ventana causal** de los últimos `W=28` días activos de ese usuario (incluyendo `t`), en orden cronológico.

- Si el usuario tiene menos de `W` días previos (incluyendo `t`), la ventana se rellena por la **izquierda con ceros** y se usa una **máscara** (`1`=día real, `0`=padding) para que el modelo y la función de pérdida ignoren el padding.
- El resultado: exactamente **330452 ventanas** (una por fila de `df`), cada una de shape `(W, F)`, cuya última posición es el día `t` que se está evaluando.
- Para no materializar un tensor de `330452 × 28 × 15` en memoria, el `Dataset` de PyTorch construye cada ventana **al vuelo** a partir de un índice por usuario (`user -> array de posiciones de fila en `X`, en orden de día`).

In [ ]:
# [compute]
# === Índices por usuario y Dataset de ventanas trailing causales ===

W = 28

users_arr = df["user"].to_numpy()

# dict user -> array de posiciones de fila (en orden de día, ya que df está sort(user, day))
user_to_rows: dict[str, np.ndarray] = {}
for u in np.unique(users_arr):
    user_to_rows[u] = np.where(users_arr == u)[0]

# Para cada fila i, qué usuario y qué posición dentro de su secuencia ocupa
row_user = users_arr
row_seq_pos = np.empty(len(df), dtype=np.int64)
for u, rows in user_to_rows.items():
    row_seq_pos[rows] = np.arange(len(rows))


class TrailingWindowDataset(torch.utils.data.Dataset):
    """Construye al vuelo, para cada fila i de df, la ventana causal de W días
    del mismo usuario terminando en i, con padding por la izquierda y máscara."""

    def __init__(self, indices: np.ndarray, X: np.ndarray, user_to_rows: dict, row_user: np.ndarray, row_seq_pos: np.ndarray, W: int):
        self.indices = indices
        self.X = X
        self.user_to_rows = user_to_rows
        self.row_user = row_user
        self.row_seq_pos = row_seq_pos
        self.W = W
        self.F = X.shape[1]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        u = self.row_user[i]
        pos = self.row_seq_pos[i]
        rows = self.user_to_rows[u]

        W = self.W
        window = np.zeros((W, self.F), dtype=np.float32)
        mask = np.zeros((W,), dtype=np.float32)

        start = max(0, pos - W + 1)
        seq_rows = rows[start:pos + 1]  # filas reales, en orden cronológico, incluye el día t
        n = len(seq_rows)

        window[W - n:] = self.X[seq_rows]
        mask[W - n:] = 1.0

        return torch.from_numpy(window), torch.from_numpy(mask), i


# Split por usuarios (85% train / 15% val)
torch.manual_seed(42)
np.random.seed(42)

all_users = np.unique(users_arr)
rng = np.random.RandomState(42)
shuffled_users = all_users.copy()
rng.shuffle(shuffled_users)

n_train_users = int(0.85 * len(shuffled_users))
train_users = set(shuffled_users[:n_train_users])
val_users = set(shuffled_users[n_train_users:])

all_indices = np.arange(len(df))
train_mask_users = np.isin(row_user, list(train_users))
val_mask_users = np.isin(row_user, list(val_users))

train_indices = all_indices[train_mask_users]
val_indices = all_indices[val_mask_users]

print(f"usuarios train/val: {len(train_users)}/{len(val_users)}")
print(f"ventanas train/val (totales): {len(train_indices)}/{len(val_indices)}")

full_dataset = TrailingWindowDataset(all_indices, X, user_to_rows, row_user, row_seq_pos, W)
train_dataset_full = TrailingWindowDataset(train_indices, X, user_to_rows, row_user, row_seq_pos, W)
val_dataset = TrailingWindowDataset(val_indices, X, user_to_rows, row_user, row_seq_pos, W)

print(f"nº ventanas dataset completo (para scoring): {len(full_dataset)}")
assert len(full_dataset) == 330452, f"esperadas 330452 ventanas, obtenidas {len(full_dataset)}"

## Arquitectura y regularización

`TransformerAnomaly` es un autoencoder de secuencias: proyecta cada vector de `F=15` features a `d_model=64`, suma una **codificación posicional aprendida** (un parámetro de shape `(1, W, 64)`), pasa la secuencia por un `TransformerEncoder` de 2 capas (4 cabezas de atención, `dim_feedforward=128`, `dropout=0.2`), y reconstruye las `F=15` features de cada posición con una capa lineal final.

Regularización explícita contra el sobreajuste, dado que cada usuario aparece en muchas ventanas solapadas:

- **Split por usuarios** (no por filas): las ventanas de un usuario van enteras a train o a val, para medir generalización a usuarios no vistos.
- **Denoising**: durante el entrenamiento, el 15% de las posiciones temporales reales de la ventana de entrada se pone a cero (ruido), pero la pérdida se calcula contra la ventana **original** sin ruido — el modelo debe aprender a "rellenar" comportamiento típico, no solo copiar la entrada.
- `dropout=0.2` dentro de las capas del encoder.
- `weight_decay=1e-4` en AdamW.
- **Early stopping** con `patience=5` sobre la pérdida de validación (sin ruido), restaurando los mejores pesos al final.

In [ ]:
# [compute]
# === Modelo: TransformerAnomaly ===

D_MODEL = 64
N_HEAD = 4
DIM_FF = 128
DROPOUT = 0.2
N_LAYERS = 2


class TransformerAnomaly(nn.Module):
    def __init__(self, n_features: int, d_model: int, n_head: int, dim_ff: int, dropout: float, n_layers: int, window: int):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos_encoding = nn.Parameter(torch.zeros(1, window, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_head, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.output_proj = nn.Linear(d_model, n_features)

    def forward(self, x):
        h = self.input_proj(x) + self.pos_encoding
        h = self.encoder(h)
        return self.output_proj(h)


model = TransformerAnomaly(F, D_MODEL, N_HEAD, DIM_FF, DROPOUT, N_LAYERS, W).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"TransformerAnomaly: {n_params:,} parámetros")

In [ ]:
# [compute]
# === Entrenamiento: denoising + early stopping por split de usuarios ===

torch.manual_seed(42)
np.random.seed(42)

if device == "cuda":
    EPOCHS, TRAIN_SUBSAMPLE = 40, None
else:
    EPOCHS, TRAIN_SUBSAMPLE = 2, 40000   # verificacion local rapida: pocas epocas y submuestreo de ventanas de train

NOISE_FRAC = 0.15
BATCH_SIZE = 256
PATIENCE = 5

if TRAIN_SUBSAMPLE is not None and TRAIN_SUBSAMPLE < len(train_indices):
    sub_rng = np.random.RandomState(42)
    sub_idx = sub_rng.choice(len(train_indices), size=TRAIN_SUBSAMPLE, replace=False)
    train_indices_used = train_indices[sub_idx]
else:
    train_indices_used = train_indices

train_dataset = TrailingWindowDataset(train_indices_used, X, user_to_rows, row_user, row_seq_pos, W)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"EPOCHS={EPOCHS}, ventanas train usadas={len(train_dataset)}, ventanas val={len(val_dataset)}")

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)


def masked_mse(recon, target, mask):
    # mask: (B, W) -> (B, W, 1) para broadcast sobre F
    m = mask.unsqueeze(-1)
    sq_err = (recon - target) ** 2 * m
    denom = m.sum() * target.shape[-1]
    return sq_err.sum() / denom.clamp(min=1.0)


best_val_loss = float("inf")
best_state = None
epochs_no_improve = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_total, train_batches = 0.0, 0
    for window, mask, _ in train_loader:
        window = window.to(device)
        mask = mask.to(device)

        # Denoising: poner a 0 el 15% de las posiciones temporales REALES de la entrada
        noise_mask = (torch.rand_like(mask) < NOISE_FRAC) & (mask > 0)
        noisy_window = window.clone()
        noisy_window[noise_mask] = 0.0

        optimizer.zero_grad()
        recon = model(noisy_window)
        loss = masked_mse(recon, window, mask)
        loss.backward()
        optimizer.step()

        train_loss_total += loss.item()
        train_batches += 1

    model.eval()
    val_loss_total, val_batches = 0.0, 0
    with torch.no_grad():
        for window, mask, _ in val_loader:
            window = window.to(device)
            mask = mask.to(device)
            recon = model(window)
            loss = masked_mse(recon, window, mask)
            val_loss_total += loss.item()
            val_batches += 1

    train_loss = train_loss_total / max(train_batches, 1)
    val_loss = val_loss_total / max(val_batches, 1)
    print(f"epoch {epoch:2d}/{EPOCHS} - train_loss: {train_loss:.5f} - val_loss: {val_loss:.5f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping en epoch {epoch} (sin mejora durante {PATIENCE} épocas)")
            break

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"Mejores pesos restaurados (val_loss={best_val_loss:.5f})")

In [ ]:
# [compute]
# === Scoring: error de reconstrucción del último día de cada ventana (330452 user-días) ===

full_loader = torch.utils.data.DataLoader(full_dataset, batch_size=BATCH_SIZE, shuffle=False)

score_transformer = np.zeros(len(df), dtype=np.float32)

model.eval()
with torch.no_grad():
    for window, mask, idx in full_loader:
        window = window.to(device)
        recon = model(window)
        mse_last = ((recon[:, -1, :] - window[:, -1, :]) ** 2).mean(dim=1)
        score_transformer[idx.numpy()] = mse_last.cpu().numpy()

print(f"score_transformer: len={len(score_transformer)}")
print(f"  min={score_transformer.min():.5f}, max={score_transformer.max():.5f}, mean={score_transformer.mean():.5f}")
print(f"  no finitos: {(~np.isfinite(score_transformer)).sum()}")

assert len(score_transformer) == 330452, f"esperadas 330452 puntuaciones, obtenidas {len(score_transformer)}"
assert np.isfinite(score_transformer).all(), "score_transformer contiene valores no finitos"

In [ ]:
# [compute]
# === Integración: añadir score_transformer a user_day_scores.parquet ===

df_scores_new = df.select("user", "day").with_columns(
    pl.Series("score_transformer", score_transformer)
)

scores_path = PROCESSED_DIR / "user_day_scores.parquet"
scores_existing = pl.read_parquet(scores_path)

scores_updated = scores_existing.join(df_scores_new, on=["user", "day"], how="left")

assert scores_updated.shape[0] == 330452, f"esperadas 330452 filas, obtenidas {scores_updated.shape[0]}"
assert scores_updated.shape[1] == 9, f"esperadas 9 columnas, obtenidas {scores_updated.shape[1]}"
assert scores_updated["score_transformer"].null_count() == 0, "nulls en score_transformer tras el join"

scores_updated.write_parquet(scores_path)

# Copia independiente solo con el nuevo score
transformer_path = PROCESSED_DIR / "user_day_transformer.parquet"
df_scores_new.write_parquet(transformer_path)

print(f"Guardado: {scores_path}")
print(f"  shape: {scores_updated.shape}")
print(f"  columnas: {scores_updated.columns}")
print(f"Guardado: {transformer_path}")
print(f"  shape: {df_scores_new.shape}")

In [ ]:
# [compute]
# === Evaluación compacta: comparativa de los 4 modelos ===

model_cols = ["score_rules", "score_iforest", "score_autoencoder", "score_transformer"]

scores_pd = scores_updated.to_pandas()
y_day = scores_pd["label_malicious_day"].to_numpy()

rows = []
for m in model_cols:
    score = scores_pd[m].to_numpy()

    ap_day = average_precision_score(y_day, score)

    # Agregación por usuario (score máximo)
    user_agg = (
        scores_updated.group_by("user")
        .agg(
            pl.col(m).max().alias("user_score"),
            pl.col("is_insider_user").any().alias("y_user"),
            pl.col("scenario").filter(pl.col("scenario") > 0).first().fill_null(0).alias("scenario"),
        )
    )
    user_score = user_agg["user_score"].to_numpy()
    y_user = user_agg["y_user"].to_numpy().astype(int)
    scenario_user = user_agg["scenario"].to_numpy()

    n_insiders = int(y_user.sum())
    order = np.argsort(-user_score)
    top70_idx = order[:n_insiders]

    det_top70_total = y_user[top70_idx].sum() / n_insiders

    det_by_scenario = {}
    for sc in (1, 2, 3):
        n_sc = int((scenario_user == sc).sum())
        if n_sc == 0:
            det_by_scenario[sc] = float("nan")
            continue
        detected_sc = ((scenario_user[top70_idx] == sc)).sum()
        det_by_scenario[sc] = detected_sc / n_sc

    rows.append({
        "modelo": m,
        "ap_day": ap_day,
        "det_top70_total": det_top70_total,
        "det_esc1": det_by_scenario[1],
        "det_esc2": det_by_scenario[2],
        "det_esc3": det_by_scenario[3],
    })

results = pl.DataFrame(rows)
print(results)
results

**Nota:** en CPU (verificación, `EPOCHS=2`, submuestreo de ventanas de train a `TRAIN_SUBSAMPLE`) estos números **no son representativos** — el modelo apenas ha visto datos y no ha convergido. Los resultados válidos para comparar `score_transformer` con los otros 3 modelos salen del **run completo en Colab con GPU** (`EPOCHS=40`, sin submuestreo).

## Conclusiones

*(Plantilla — completar tras el run en Colab GPU)*

- **¿Mejora el Transformer la detección del escenario 2 frente al autoencoder (9/30 ≈ 0.300)?** Comparar `det_esc2` de `score_transformer` contra `score_autoencoder` en la tabla anterior. La hipótesis de diseño es que sí, porque el Transformer modela la **trayectoria** de cada usuario (ventana de 28 días) en lugar de evaluar cada día de forma aislada, y el escenario 2 se caracteriza por una escalada gradual de actividad sospechosa (USB, ritmo de copias) antes de la salida del empleado.
- **Escenarios 1 y 3**: comparar también `det_esc1` y `det_esc3` — el escenario 1 (fuga puntual a wikileaks tras un cambio brusco de patrón) y el escenario 3 (keylogger + mass email) pueden seguir siendo mejor capturados por modelos que reaccionan a un día concreto (autoencoder, reglas).
- **AUC-PR a nivel usuario-día (`ap_day`)**: indica si el Transformer también mejora la localización temporal exacta del día anómalo, no solo la identificación del usuario.
- **Implicación para el dashboard (fase 5)**: si el Transformer resulta claramente superior en escenario 2 pero no en 1/3, la integración debería presentar **el mejor detector por tipo de amenaza** (o una combinación/ensemble de scores), en lugar de un único modelo "ganador" global.